In [0]:
df = spark.sql("""
SELECT
  b.branch_id,
  b.branch_city,
  UPPER(TRIM(b.branch_country)) AS branch_country
FROM policyprojcatalog.policyprojdb.branch b
WHERE b.branch_id IS NOT NULL
  AND b.merge_flag = false
""")

display(df)

In [0]:
df.createOrReplaceTempView("clean_branch")

spark.sql("""
MERGE INTO policyprojcatalog.silver.branch AS T
USING clean_branch AS S
ON T.branch_id = S.branch_id

WHEN MATCHED THEN UPDATE SET
  T.branch_country = S.branch_country,
  T.branch_city = S.branch_city,
  T.merged_timestamp = current_timestamp()

WHEN NOT MATCHED THEN INSERT (
  branch_id,
  branch_country,
  branch_city,
  merged_timestamp
)
VALUES (
  S.branch_id,
  S.branch_country,
  S.branch_city,
  current_timestamp()
)
""")

In [0]:
spark.sql("""
UPDATE policyprojcatalog.policyprojdb.branch
SET merge_flag = true
WHERE merge_flag = false
""")